### Asymptotic Verification of Appendix A

As a sanity check, we verify some of the less obvious results of the appendix. Namely, the asymptotic exponents, the prefactors, the beta-binomial mixture identites, $\alpha_{\rm th}=1$, and the logarithmic
correction at $\alpha_{\rm th}=\alpha_d+2$.

Additionally, a few finite-$n$ numerical confirmations at one representative parameter point close each cell.
Every check prints a `PASS`/`FAIL`, indicating whether or not the statements are symbolically/numerically equivalent.

In [ ]:
# Import statments
import sympy as sp
import numpy as np
from scipy.special import betainc, betaln, gammaln, zeta

# Representative point for the finite-n numerical checks (same as Figure 5).
ad, bd, bth, phibar = 0.75, 3.0, 3.0, 0.20
md = ad / (ad + bd)
tc = ad + 2.0

# Initialize the test results.
RESULTS = []
def ok(name, cond):
    RESULTS.append(bool(cond))
    print(("PASS " if cond else "FAIL ") + name)

### Asymptotic behavior of the threshold distribution (A1)

Here, we look at equation A6, the prefactor $C_{th}$ of the threshold distribution asymptotics.

In [2]:
# Define the variables
t, u = sp.symbols('t u', positive=True) # t is integrated over in RIBF, u is the limit.
al, be, ph = sp.symbols('alpha_th beta_th phi_bar', positive=True) # a_th, b_th, and phi_bar. 
m = al/(al + be)
k = sp.symbols('k', positive=True) # Degree k.


Iu = u**al/(al*sp.beta(al, be))*sp.hyper((al, 1 - be), (al + 1,), u) # Equation A4, the standard hypergeometric 2F1 expansion of the RIBF.

Fk = Iu.subs(u, m/(k*ph)) # u-substitution
Cth = m**al/(al*sp.beta(al, be)*ph**al) # Our claimed Cth

# Series expansion check (Equation A4)
ser = sp.series((1 - t)**(be - 1), t, 0, 2).removeO()
Iu2 = sp.integrate(t**(al - 1)*ser, (t, 0, u))/sp.beta(al, be)
expansion = u**al/(al*sp.beta(al, be))*(1 + al*(1 - be)/(al + 1)*u)

ok('Eq. A4 I_u small t expansion is as claimed',
   sp.simplify(Iu2 - expansion) == 0)

ok('Eq. A6 F(1/k)/(C_th k^-alpha_th) -> 1 (C_th is as claimed)',
   sp.simplify(sp.limit(Fk/(Cth*k**(-al)), k, sp.oo)) == 1) # lim k->infty of F(1/k) / Cth k^(-a_th) = 1

PASS Eq. A4 I_u small t expansion is as claimed
PASS Eq. A6 F(1/k)/(C_th k^-alpha_th) -> 1 (C_th is as claimed)


### The cascade parameter on Erdős–Rényi graphs (A2)

Next, we verify appendix section 2 on ER graphs. We want to verify that equation A9, A11, and A14 are correct, which are the scaling, the expression for $F(1/k)$, and the critical average threhsold expressions, respectively.

In [3]:
I1 = sp.integrate((1 - t)**(be - 1), (t, 0, u))/sp.beta(1, be) # integral of Equation A10 
m1 = sp.Integer(1)/(1 + be) 
F1 = I1.subs(u, m1/(k*ph)) # u-substitution for equation A11

# sum_k e^-z z^k/k! k(k-1)k^-at = z^2 E[(K+2)^-at] ~ z^2 z^-at, so kappa ~ (1/z) z^(2-at) by degree concentration.
zs = sp.Symbol('z', positive=True)
krate = (zs**2*zs**(-al))/zs 

ok('Eq. A9 ER leading power = z^(1-alpha_th)', sp.simplify(krate/zs**(1 - al)) == 1)

ok('Eq. A11 is asymptotically a claimed for alpha=1,  kF(1/k) -> beta/((1+beta)phi)',
   sp.simplify(sp.limit(k*F1, k, sp.oo) - be/((1 + be)*ph)) == 0)

ok('Eq. A14 phi_crit = beta_th/(1+beta_th) (the critical mean threshold is as claimed)', # trivial
   sp.simplify(sp.solve(sp.Eq(be/((1 + be)*ph), 1), ph)[0] - be/(1 + be)) == 0)


PASS Eq. A9 ER leading power = z^(1-alpha_th)
PASS Eq. A11 is asymptotically a claimed for alpha=1,  kF(1/k) -> beta/((1+beta)phi)
PASS Eq. A14 phi_crit = beta_th/(1+beta_th) (the critical mean threshold is as claimed)


### Threshold-dominated regime ($\alpha_{th} < \alpha_d + 2$) (A4a)

We want to verify Equations A26 and A27, which dictate the scaling of $\kappa(z)$ and the coefficient $C_1$, respectively.

In [4]:
# Define the parameters for the degree distribution.
ad_, bd_, nn, kk = sp.symbols('alpha_d beta_d n k', positive=True)
z, at = sp.symbols('z alpha_th', positive=True)
n = z/(ad_/(ad_ + bd_)) # n is z/md
tau = sp.symbols('tau', positive=True) # k/n

# Equation A6/A21 
Cth1 = (at/(at + be))**at/(at*sp.beta(at, be)*ph**at)

# Equation A23 with A24 mixing moment. 
kapp1 = (Cth1/z)*n**(2 - at)*sp.beta(ad_ + 2 - at, bd_)/sp.beta(ad_, bd_) 

C1_derived = kapp1/z**(1 - at) # Strip kappa of its factor of z^(1-a), resulting in C1
C1_paper = Cth1*(ad_/(ad_ + bd_))**(at - 2)*sp.beta(ad_ + 2 - at, bd_)/sp.beta(ad_, bd_) # Equation A27

ok('Eq. A26 C1 scaling is kappa ~ z^(1-alpha_th)',
   sp.simplify(kapp1/(C1_paper*z**(1 - at))) == 1)

ok('Eq. A27 C1 prefactor matches stated form', sp.simplify(C1_derived/C1_paper) == 1)

PASS Eq. A26 C1 scaling is kappa ~ z^(1-alpha_th)
PASS Eq. A27 C1 prefactor matches stated form


### Degree-dominated regime ($\alpha_{th} > \alpha_d + 2$) (A4b)

We want to verify Equations A29, A30, A31, and A32.

In [5]:
# (a very ugly) Equation A28 in terms of gamma functions.
P_gamma = (sp.gamma(nn + 1)/(sp.gamma(kk + 1)*sp.gamma(nn - kk + 1))
           * sp.gamma(kk + ad_)*sp.gamma(nn - kk + bd_)*sp.gamma(ad_ + bd_)
           / (sp.gamma(nn + ad_ + bd_)*sp.gamma(ad_)*sp.gamma(bd_)))

# Equation A29, the large n asymptotics
rhs = (nn**(-ad_)*sp.gamma(ad_ + bd_)/sp.gamma(bd_)
       * sp.gamma(kk + ad_)/(sp.gamma(ad_)*sp.gamma(kk + 1)))

ok('Eq. A29 fixed-k pmf: P/rhs -> 1 (large n asymptotics are as stated)', sp.simplify(sp.limit(P_gamma/rhs, nn, sp.oo)) == 1)

# Degree-dominated: amplitude Gamma(ad+bd)/Gamma(bd) * md^ad, exponent z^-(ad+1).
# P(k)~n^-ad Gamma(ad+bd)/Gamma(bd) Gamma(k+ad)/(Gamma(ad)k!); kappa=(1/z) sum k(k-1)P F.

# Equation A19 substituted for asymptote A29
amp = (1/z)*n**(-ad_)*sp.gamma(ad_ + bd_)/sp.gamma(bd_)

# This is just saying that the functions that are not a function of k in Eq. A31 are asymptotically m_d^a_d * z^-(a_d +1)
ok('Eq. A31 scaling is md^ad z^-(ad+1)',
   sp.simplify(amp/((ad_/(ad_ + bd_))**ad_*z**(-(ad_ + 1))*sp.gamma(ad_ + bd_)/sp.gamma(bd_))) == 1)

# Summand large-k law k^(1+ad-at) and summability at > ad+2.
ok('Eq. A32 C2 summand exponent is 1+alpha_d-alpha_th',
   sp.simplify((ad_ - 1) + (2 - at) - (1 + ad_ - at)) == 0)

ok('C2 summable iff alpha_th > alpha_d+2',
   sp.simplify(sp.solve(sp.Eq(1 + ad_ - at, -1), at)[0] - (ad_ + 2)) == 0)


PASS Eq. A29 fixed-k pmf: P/rhs -> 1 (large n asymptotics are as stated)
PASS Eq. A31 scaling is md^ad z^-(ad+1)
PASS Eq. A32 C2 summand exponent is 1+alpha_d-alpha_th
PASS C2 summable iff alpha_th > alpha_d+2


###  Marginal regime ($\alpha_{th} = \alpha_d + 2$) (A4c)

Lastly, we wish to show A34 and A35 to be true, which are that the scaling is $\ln z$ and that the prefactor is $C_c$, respectively.

In [6]:
# Define the marginal condition
atb = ad_ + 2
md_ = ad_/(ad_ + bd_)
Cth_b = (atb/(atb + be))**atb/(atb*sp.beta(atb, be)*ph**atb)   # C_th at the boundary
nb = z/md_

# kappa for the marginal case.
kapp_m = (1/z)*nb**(-ad_)*sp.gamma(ad_ + bd_)/sp.gamma(bd_)*(Cth_b/sp.gamma(ad_))*sp.log(nb)

# Deriving A35
Cc_derived = kapp_m/(z**(-(ad_ + 1))*sp.log(nb))

# Stated A35
Cc_paper = Cth_b*md_**ad_/sp.beta(ad_, bd_) 

ok('Eq. A34 Cc exponent scales as kappa ~ z^-(ad+1) ln z',
   sp.simplify(kapp_m/(Cc_paper*z**(-(ad_ + 1))*sp.log(nb))) == 1)

ok('Eq. A35 Cc prefactor matches stated form', sp.simplify(Cc_derived/Cc_paper) == 1)


PASS Eq. A34 Cc exponent scales as kappa ~ z^-(ad+1) ln z
PASS Eq. A35 Cc prefactor matches stated form


--- 



### Numerical checks


For fun, just numerics on a single choice of parameters to show that these converge.

Parameters are arbitrary, as this holds of all of them. We pick the point $\alpha_d=0.75,\ \beta_d=3,\ \beta_{\rm th}=3,\ \bar\phi=0.2$.

In [7]:
def Cth_f(t):
    m = t/(t + bth)
    return np.exp(t*np.log(m) - np.log(t) - betaln(t, bth) - t*np.log(phibar))

def C1_f(t):
    return Cth_f(t)*md**(t - 2)*np.exp(betaln(ad + 2 - t, bd) - betaln(ad, bd))

def Cc_f():
    return Cth_f(tc)*md**ad/np.exp(betaln(ad, bd))

def bb_pmf(n, a=ad, b=bd):
    p = np.empty(n + 1)
    p[0] = np.exp(betaln(a, n + b) - betaln(a, b))
    j = np.arange(n, dtype=float)
    p[1:] = p[0]*np.cumprod(((n - j)*(j + a))/((j + 1)*(n - j - 1 + b)))
    return p

def Fk(k, t):
    m = t/(t + bth)
    return betainc(t, bth, np.minimum(1.0, m/(phibar*k)))

def kappa(n, t):
    p = bb_pmf(n)
    k = np.arange(2, n + 1, dtype=float)
    return np.dot(k*(k - 1)*p[2:], Fk(k, t))/(n*md)

def S_exact(t, cut=1_000_000):
    k = np.arange(2, cut + 1, dtype=float)
    terms = k*(k - 1)*np.exp(gammaln(k + ad) - gammaln(k + 1))*Fk(k, t)
    return terms.sum() + Cth_f(t)*zeta(t - ad - 1, cut + 1)

def C2_f(t, cut=1_000_000):
    return md**ad/np.exp(betaln(ad, bd))*S_exact(t, cut)


In [10]:
nbig = 10**8
zbig = nbig*md

r1 = kappa(nbig, 1.5)/(C1_f(1.5)*zbig**(1 - 1.5))
ok(f'numeric kappa~C1 z^(1-at): ratio={r1:.6f}', abs(r1 - 1) < 1e-3)

r2 = kappa(nbig, 4.0)/(C2_f(4.0)*zbig**(-(ad + 1)))
ok(f'numeric kappa~C2 z^-(ad+1): ratio={r2:.6f}', abs(r2 - 1) < 1e-3)

consts = []
for n in [10**4, 10**5, 10**6]:
    z = n*md
    ratio = kappa(n, tc)/(Cc_f()*z**(-(ad + 1))*np.log(z/md))
    consts.append((1 - ratio)*np.log(n))
ok(f'marginal (1-ratio)ln n constant has a spread of {max(consts)-min(consts):.4f}, hence convergent.',
   max(consts) - min(consts) < 0.01)

PASS numeric kappa~C1 z^(1-at): ratio=1.000000
PASS numeric kappa~C2 z^-(ad+1): ratio=0.999999
PASS marginal (1-ratio)ln n constant has a spread of 0.0065, hence convergent.


In [9]:
print(f'{sum(RESULTS)}/{len(RESULTS)} checks passed')
assert all(RESULTS)

16/16 checks passed
